In [0]:
USE CATALOG ieee_cis;
USE SCHEMA bronze;

# Bronze Data Ingestion

In [0]:
CREATE OR REFRESH STREAMING TABLE ieee_cis.bronze.train_transaction
COMMENT "Raw train transactions from the Kaggle CSV"
AS SELECT
  *,
  _metadata.file_path AS _source_file,
  current_timestamp() AS _ingested_at
FROM STREAM read_files(
  '/Volumes/ieee_cis/bronze/raw/train_transaction/',
  format => 'csv',
  header => true
);

In [0]:
CREATE OR REFRESH STREAMING TABLE ieee_cis.bronze.test_transaction
COMMENT "Raw train transactions from the Kaggle CSV"
AS SELECT
  *,
  _metadata.file_path AS _source_file,
  current_timestamp() AS _ingested_at
FROM STREAM read_files(
  '/Volumes/ieee_cis/bronze/raw/test_transaction/',
  format => 'csv',
  header => true
);

In [0]:
CREATE OR REFRESH STREAMING TABLE ieee_cis.bronze.train_identity
COMMENT "Raw train transactions from the Kaggle CSV"
AS SELECT
  *,
  _metadata.file_path AS _source_file,
  current_timestamp() AS _ingested_at
FROM STREAM read_files(
  '/Volumes/ieee_cis/bronze/raw/train_identity/',
  format => 'csv',
  header => true
);

In [0]:
CREATE OR REFRESH STREAMING TABLE ieee_cis.bronze.test_identity
COMMENT "Raw train transactions from the Kaggle CSV"
AS SELECT
  *,
  _metadata.file_path AS _source_file,
  current_timestamp() AS _ingested_at
FROM STREAM read_files(
  '/Volumes/ieee_cis/bronze/raw/test_identity/',
  format => 'csv',
  header => true
);

# Silver Transformation 

## Understanding the columns

`train_transaction.csv` has **394 columns**. Vesta, the company that released the data, deliberately hid what most of them mean to protect customer privacy. What's known comes from their official description on Kaggle. Here they are by group:

| Column(s) | Count | Meaning | Example values |
|---|---|---|---|
| `TransactionID` | 1 | Unique ID of the transaction. It's also the key for joining to `train_identity.csv`. | 2987000 |
| `isFraud` | 1 | **The label: 1 = fraud, 0 = legit.** | 0 / 1 |
| `TransactionDT` | 1 | Seconds since a hidden start date, **not a real timestamp**. It starts at 86,400 (1 day) and covers about 6 months. | 86400 |
| `TransactionAmt` | 1 | Payment amount in US dollars | 68.5 |
| `ProductCD` | 1 | Product code. It may be a service rather than a physical product. | W, C, R, H, S |
| `card1`–`card6` | 6 | Payment card info: card type, card category, issuing bank, country, and so on | |
| ↳ `card4` | | Card network | visa, mastercard, american express, discover |
| ↳ `card6` | | Card type | debit, credit |
| ↳ `card1`, `card2`, `card3`, `card5` | | Hidden codes. `card1` roughly identifies a card. | 13926 |
| `addr1` | 1 | Purchaser's billing **region** | 315 |
| `addr2` | 1 | Purchaser's billing **country** | 87 |
| `dist1`, `dist2` | 2 | Distances between things like the billing address, mailing address, zip code, IP address and phone area | 19.0 |
| `P_emaildomain` | 1 | **Purchaser's** email domain | gmail.com |
| `R_emaildomain` | 1 | **Recipient's** email domain. Often empty, because many transactions have no recipient. | hotmail.com |
| `C1`–`C14` | 14 | **Counts**, for example how many addresses are linked to this card. The exact meaning of each is hidden. | 1.0 |
| `D1`–`D15` | 15 | **Time gaps in days**, for example days since this card's previous transaction | 14.0 |
| `M1`–`M9` | 9 | **Match checks**, for example "does the name on the card match the address?" | T, F (`M4`: M0/M1/M2) |
| `V1`–`V339` | 339 | **Vesta's own engineered features**: rankings, counts and relationships between entities, such as "how often this card appeared with this email in 24 hours" | Numbers |

## How fraud was labelled
A transaction is marked fraud if its card had a **chargeback** (the cardholder disputed the payment). **Later transactions linked to the same account, email or billing address are marked fraud too.** If nothing was reported within 120 days, the transaction counts as legit. So fraud labels follow a *person* over time, not a single purchase.

## Things that matter when you build
1. **Some numbers are really categories.** `card1`, `card2`, `card3`, `card5`, `addr1` and `addr2` look like numbers, but they're codes. `addr1 = 315` isn't "bigger" than `addr1 = 204`. Tree models cope with this, but keep it in mind.
2. **Lots of blanks.** Many columns, especially `dist2`, the D columns and the V columns, are mostly empty. That's expected, not an error, and as Chapter 2 explains, a blank can itself be a fraud signal.
3. **A trick from the competition.** Top Kaggle competitors noticed that `D1` appears to be "days since the card was first used". That means `TransactionDT / 86400 − D1` stays about the same for one card, and they used it to tell individual customers apart. It's a good idea to try once your baseline model works.

Source: [Vesta's data description on Kaggle](https://www.kaggle.com/c/ieee-fraud-detection/discussion/101203)

## Shared features: train and test in one view
`silver.transactions` defines every feature once, over train and test together, so the model always sees the same
feature at scoring time as in training. The windows only look at earlier rows: a training row never sees test data,
while a test row can use the customer's history from the training period. `silver.train` and `silver.test` are
filters of it.

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW silver.transactions
COMMENT "Train + test transactions with one shared feature definition"
AS
WITH base AS (
  -- the same columns from both halves; test has no label
  SELECT t.TransactionID, t.TransactionDT, t.isFraud, TRUE AS is_train,
         t.TransactionAmt, t.ProductCD,
         t.card1, t.card2, t.card3, t.card4, t.card5, t.card6,
         t.addr1, t.addr2, t.dist1, t.P_emaildomain, t.R_emaildomain,
         t.C1, t.C2, t.C3, t.C4, t.C5, t.C6, t.C7, t.C8, t.C9, t.C10, t.C11, t.C12, t.C13, t.C14,
         t.D1, t.D4, t.D10, t.D15, t.M4, t.M6,
         t.V95, t.V96, t.V97, t.V126, t.V127, t.V128, t.V279, t.V280, t.V306, t.V307, t.V308,   -- Vesta's per-user velocity counts and amounts
         i.DeviceType                                   -- from the identity table; NULL for ~76% of rows
  FROM bronze.train_transaction t
  LEFT JOIN bronze.train_identity i ON t.TransactionID = i.TransactionID

  UNION ALL

  SELECT t.TransactionID, t.TransactionDT, NULL AS isFraud, FALSE AS is_train,
         t.TransactionAmt, t.ProductCD,
         t.card1, t.card2, t.card3, t.card4, t.card5, t.card6,
         t.addr1, t.addr2, t.dist1, t.P_emaildomain, t.R_emaildomain,
         t.C1, t.C2, t.C3, t.C4, t.C5, t.C6, t.C7, t.C8, t.C9, t.C10, t.C11, t.C12, t.C13, t.C14,
         t.D1, t.D4, t.D10, t.D15, t.M4, t.M6,
         t.V95, t.V96, t.V97, t.V126, t.V127, t.V128, t.V279, t.V280, t.V306, t.V307, t.V308,   -- Vesta's per-user velocity counts and amounts
         i.DeviceType
  FROM bronze.test_transaction t
  LEFT JOIN bronze.test_identity i ON t.TransactionID = i.TransactionID
),

keyed AS (
  SELECT *,
         timestampadd(SECOND, TransactionDT, TIMESTAMP '2017-12-01') AS event_ts,
         FLOOR(TransactionDT / 86400)                                  AS day,
         -- customer key: card + billing region + the day the card was first used (NULL parts are skipped)
         CONCAT_WS('_',
                   CAST(card1 AS STRING),
                   CAST(addr1 AS STRING),
                   CAST(FLOOR(TransactionDT / 86400) - D1 AS STRING)) AS cust_key
  FROM base
)

SELECT *,
  -- D columns as fixed dates instead of "days ago"
  day - D1  AS D1n,
  day - D4  AS D4n,
  day - D10 AS D10n,
  day - D15 AS D15n,

  HOUR(event_ts)                         AS hour,
  TransactionAmt - FLOOR(TransactionAmt) AS amt_cents,   -- odd cents often mean a foreign-currency card

  -- velocity and customer history: earlier rows only, so no leakage
  COUNT(*)            OVER card_24h  AS card_txn_24h,
  COUNT(*)            OVER cust_past AS cust_prev_txns,
  AVG(TransactionAmt) OVER cust_past AS cust_prev_amt_mean,
  TransactionAmt / NULLIF(AVG(TransactionAmt) OVER cust_past, 0) AS amt_vs_cust_mean,
  TransactionDT - LAG(TransactionDT) OVER (PARTITION BY cust_key ORDER BY TransactionDT) AS secs_since_cust_prev

FROM keyed
WINDOW
  card_24h  AS (PARTITION BY card1    ORDER BY TransactionDT RANGE BETWEEN 86400 PRECEDING AND 1 PRECEDING),
  cust_past AS (PARTITION BY cust_key ORDER BY TransactionDT ROWS  BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING);

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW silver.train (
  CONSTRAINT valid_id     EXPECT (TransactionID IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT valid_amount EXPECT (TransactionAmt > 0)        ON VIOLATION DROP ROW,
  CONSTRAINT has_card     EXPECT (card1 IS NOT NULL)          -- warn only
)
COMMENT "Clean training data: transactions joined with identity, plus shared features"
AS SELECT * EXCEPT (is_train)
FROM silver.transactions
WHERE is_train;

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW silver.test (
  -- warn only: every test transaction must be scored, so nothing is dropped
  CONSTRAINT valid_id     EXPECT (TransactionID IS NOT NULL),
  CONSTRAINT valid_amount EXPECT (TransactionAmt > 0),
  CONSTRAINT has_card     EXPECT (card1 IS NOT NULL)
)
COMMENT "Clean test data (no label): transactions joined with identity, plus shared features"
AS SELECT * EXCEPT (is_train, isFraud)
FROM silver.transactions
WHERE NOT is_train;